Topic: PySpark Join - Customers and Orders

Objective:
Join customer data with order data to create a customer order summary.


In [0]:

from pyspark.sql import SparkSession
from pyspark.sql.functions import col, round, sum, count


spark = SparkSession.builder.appName("daily-practice").getOrCreate()


CUSTOMERS_PATH = "/Volumes/workspace/data/daily_practice/data/customers_join.csv"
ORDERS_PATH = "/Volumes/workspace/data/daily_practice/data/orders_join.csv"
OUTPUT_PATH = "/Volumes/workspace/data/daily_practice/output/customer_order_summary"


customers_df = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(CUSTOMERS_PATH)
)

orders_df = (
    spark.read
    .option("header", True)
    .option("inferSchema", True) 
    .csv(ORDERS_PATH)
)


print("Customers Data")
display(customers_df)

print("Orders Data")
display(orders_df)




Customers Data


customer_id,customer_name,city
1,Avishek Bhandari,Detroit
2,John Smith,Chicago
3,Sara Khan,New York
4,Maria Lopez,Dallas
5,David Lee,Boston


Orders Data


order_id,customer_id,order_amount,order_status
1001,1,1200.5,Completed
1002,2,750.0,Completed
1003,1,300.0,Cancelled
1004,3,45.75,Completed
1005,4,600.25,Completed
1006,6,999.99,Completed
1007,1,1000.5,Completed


In [0]:
completed_orders_df = (
    orders_df
    .filter(col("order_status") == "Completed")
)


customer_orders_df = (
    customers_df.alias("c")
    .join(
        completed_orders_df.alias("o"),
        col("c.customer_id") == col("o.customer_id"),
        "inner"
    )
    .select(
        col("c.customer_id"),
        col("c.customer_name"),
        col("c.city"),
        col("o.order_id"),
        col("o.order_amount"),
        col("o.order_status")
    )
)


print("Joined Customer Orders")
display(customer_orders_df)



Joined Customer Orders


customer_id,customer_name,city,order_id,order_amount,order_status
1,Avishek Bhandari,Detroit,1001,1200.5,Completed
2,John Smith,Chicago,1002,750.0,Completed
3,Sara Khan,New York,1004,45.75,Completed
4,Maria Lopez,Dallas,1005,600.25,Completed
1,Avishek Bhandari,Detroit,1007,1000.5,Completed


In [0]:
customer_summary_df = (
    customer_orders_df
    .groupBy("customer_id", "customer_name", "city")
    .agg(
        count("order_id").alias("total_completed_orders"),
        round(sum("order_amount"), 2).alias("total_order_amount")
    )
    .orderBy(col("total_order_amount").desc())
)


print("Customer Order Summary")
display(customer_summary_df)


(
    customer_summary_df.write
    .mode("overwrite")
    .parquet(OUTPUT_PATH)
)


print(f"Customer order summary written successfully to: {OUTPUT_PATH}")

Customer Order Summary


customer_id,customer_name,city,total_completed_orders,total_order_amount
1,Avishek Bhandari,Detroit,2,2201.0
2,John Smith,Chicago,1,750.0
4,Maria Lopez,Dallas,1,600.25
3,Sara Khan,New York,1,45.75


Customer order summary written successfully to: /Volumes/workspace/data/daily_practice/output/customer_order_summary
